# 02 - 数据加载与预处理

## 学习目标
- 掌握 sklearn 内置数据集的加载方式
- 理解数据预处理的必要性
- 学会使用 StandardScaler、LabelEncoder 等工具
- 掌握 train_test_split 的正确用法

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

## 1. sklearn 内置数据集

In [ ]:
# 分类数据集
iris = datasets.load_iris()
print(f'Iris 数据集: {iris.data.shape}, 类别: {iris.target_names}')

breast_cancer = datasets.load_breast_cancer()
print(f'Breast Cancer: {breast_cancer.data.shape}, 类别: {breast_cancer.target_names}')

digits = datasets.load_digits()
print(f'Digits: {digits.data.shape}, 类别数: {len(digits.target_names)}')

# 回归数据集
california = datasets.fetch_california_housing()
print(f'California Housing: {california.data.shape}')

## 2. 数据集划分

In [ ]:
# 基本划分
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target,
    test_size=0.3,       # 30% 作为测试集
    random_state=42,     # 固定随机种子保证可复现
    stratify=iris.target # 分层抽样，保持类别比例
)

print(f'训练集: {X_train.shape}')
print(f'测试集: {X_test.shape}')
print(f'\n训练集类别分布: {np.bincount(y_train)}')
print(f'测试集类别分布: {np.bincount(y_test)}')

## 3. 特征标准化

In [ ]:
# StandardScaler: 均值为0，标准差为1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)        # 只 transform！

print('标准化前:')
print(f'  均值: {X_train.mean(axis=0).round(2)}')
print(f'  标准差: {X_train.std(axis=0).round(2)}')

print('\n标准化后:')
print(f'  均值: {X_train_scaled.mean(axis=0).round(4)}')
print(f'  标准差: {X_train_scaled.std(axis=0).round(4)}')

In [ ]:
# MinMaxScaler: 缩放到 [0, 1]
minmax = MinMaxScaler()
X_train_minmax = minmax.fit_transform(X_train)

print(f'MinMax 范围: [{X_train_minmax.min():.2f}, {X_train_minmax.max():.2f}]')

## 4. 缺失值处理

In [ ]:
# 创建带缺失值的数据
X_with_nan = X_train.copy().astype(float)
rng = np.random.RandomState(42)
mask = rng.random(X_with_nan.shape) < 0.1  # 10% 缺失
X_with_nan[mask] = np.nan

print(f'缺失值数量: {np.isnan(X_with_nan).sum()}')

# 用中位数填充
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_with_nan)

print(f'填充后缺失值: {np.isnan(X_imputed).sum()}')

## 5. 类别编码

In [ ]:
# Label Encoding (有序类别)
le = LabelEncoder()
colors = ['red', 'blue', 'green', 'red', 'blue']
encoded = le.fit_transform(colors)
print(f'Label Encoding: {colors} -> {encoded}')

# One-Hot Encoding (无序类别)
ohe = OneHotEncoder(sparse_output=False)
encoded_ohe = ohe.fit_transform(np.array(colors).reshape(-1, 1))
print(f'\nOne-Hot Encoding:')
print(f'  类别: {ohe.categories_}')
print(f'  编码:\n{encoded_ohe}')

## 6. 完整预处理流程

In [ ]:
# 用 DataFrame 演示完整流程
df = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 35, 28, np.nan, 50],
    'income': [50000, 60000, 70000, np.nan, 80000, 55000, 65000, 90000],
    'city': ['Beijing', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai', 'Beijing', 'Guangzhou', 'Shanghai'],
    'purchased': [0, 1, 0, 1, 1, 0, 1, 1]
})

print('原始数据:')
print(df)

# Step 1: 分离特征和标签
X = df.drop('purchased', axis=1)
y = df['purchased']

# Step 2: 处理缺失值
num_cols = ['age', 'income']
imputer = SimpleImputer(strategy='median')
X[num_cols] = imputer.fit_transform(X[num_cols])

# Step 3: 类别编码
X = pd.get_dummies(X, columns=['city'], drop_first=True)

# Step 4: 标准化
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print('\n处理后:')
print(X_scaled.round(2))

## ⚠️ 常见错误

1. **数据泄漏**: 在测试集上 fit scaler
2. **忘记分层抽样**: 类别不平衡时不用 stratify
3. **不处理缺失值**: 很多模型不支持 NaN

---
下一个 Notebook: [03_linear_regression.ipynb](03_linear_regression.ipynb)